# N-Body CUDA / Barnes-Hut

Run cells top to bottom. After the first run, re-run just the **pull + build** cell whenever the code changes, then re-run whatever test cell you need — no need to re-clone.

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO_URL = "https://github.com/Pratik740/nbody-simulation.git"
BRANCH = "cuda-barnes-hut"
REPO_DIR = "/content/nbody-simulation"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
%cd {REPO_DIR}/cuda
!bash build.sh

## Correctness check (small n)
Compares CPU reference vs GPU brute-force vs GPU Barnes-Hut on the same initial conditions. Brute-force should match the CPU reference to float-rounding precision; Barnes-Hut should be close but not identical (that gap is the theta approximation).

In [ ]:
!./build/nbody_cuda --mode=compare --scenario=orbit --n=3 --steps=500 --dt=0.01 --theta=0.5 --print-final

In [ ]:
!./build/nbody_cuda --mode=compare --scenario=random --n=500 --steps=100 --dt=0.005 --theta=0.5

## Benchmark: brute-force O(n^2) vs Barnes-Hut O(n log n)
Watch the ms/step column grow much faster for brute-force as n increases.

In [ ]:
for n in [1000, 5000, 20000, 100000]:
    print(f"--- n={n} ---")
    !./build/nbody_cuda --mode=bruteforce --scenario=random --n={n} --steps=20 --dt=0.005
    !./build/nbody_cuda --mode=barnes_hut --scenario=random --n={n} --steps=20 --dt=0.005 --theta=0.5